**Install:** `pip install -U langchain langchain-openai langgraph`

# 🎯 7. RAG-Powered Agents

In notebooks 01-06 of our RAG lectures series, we built sophisticated retrieval systems. Now we **connect them to agents** — creating agents that can search knowledge bases, evaluate results, and reason with retrieved evidence.

In this notebook:

1. **RAG recap** — the core retrieval pipeline
2. **From static to agentic RAG** — what changes when agents drive retrieval
3. **Retrieval as a tool** — wrapping vector search in a LangChain tool
4. **Building a RAG-powered agent** with LangGraph
5. **Iterative retrieval** — the maker-checker pattern
6. **Hybrid retrieval** — combining BM25 + vector search

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
# if not os.environ.get("OPENAI_API_KEY"):
#     import getpass
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"
EMBED_MODEL = 'text-embedding-3-small'

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.documents import Document
from langchain_chroma import Chroma

llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
embedder = OpenAIEmbeddings(model=EMBED_MODEL)
print(f'Model: {LLM_MODEL} | Embeddings: {EMBED_MODEL}')

Model: gpt-4o-mini | Embeddings: text-embedding-3-small


## 7.1 Setting Up the Knowledge Base

We'll create a domain-specific knowledge base about AI agent frameworks:

In [3]:
# Knowledge base about AI agent frameworks
knowledge_base = [
    Document(page_content='LangGraph is a framework by LangChain for building stateful, graph-based agent workflows. It uses StateGraph with nodes and edges for agent orchestration. Key features: conditional edges, checkpointing, human-in-the-loop.', metadata={'source': 'frameworks.txt', 'topic': 'langgraph'}),
    Document(page_content='CrewAI is a framework for building role-based multi-agent systems. Each agent has a role, goal, and backstory. Agents collaborate through structured task delegation and shared context.', metadata={'source': 'frameworks.txt', 'topic': 'crewai'}),
    Document(page_content='AutoGen by Microsoft enables multi-agent conversation patterns. It supports group chat, agent handoffs, and distributed execution via gRPC. Agents can run on separate machines.', metadata={'source': 'frameworks.txt', 'topic': 'autogen'}),
    Document(page_content='The ReAct pattern combines reasoning and acting in an iterative loop. The agent generates a Thought, takes an Action via a tool, receives an Observation, and loops until reaching a final answer.', metadata={'source': 'patterns.txt', 'topic': 'react'}),
    Document(page_content='Function calling allows LLMs to invoke external tools by generating structured JSON describing the function name and arguments. OpenAI, Anthropic, and Google all support this pattern.', metadata={'source': 'patterns.txt', 'topic': 'tools'}),
    Document(page_content='Agent memory includes short-term (conversation history), long-term (persisted facts), and episodic (past experience). LangGraph supports checkpointing with SQLite and MemorySaver.', metadata={'source': 'patterns.txt', 'topic': 'memory'}),
    Document(page_content='MCP (Model Context Protocol) by Anthropic standardizes how LLMs connect to external tools and data. It uses a client-server architecture with three primitives: Tools, Resources, and Prompts.', metadata={'source': 'protocols.txt', 'topic': 'mcp'}),
    Document(page_content='A2A (Agent-to-Agent Protocol) by Google enables inter-agent communication. Components include Agent Cards (capability discovery), Executors (task processing), and Artifacts (work products).', metadata={'source': 'protocols.txt', 'topic': 'a2a'}),
    Document(page_content='Production agents require error handling (retry with backoff), cost optimization (model routing, caching), security (input validation, PII redaction), and observability (tracing with LangSmith or Langfuse).', metadata={'source': 'production.txt', 'topic': 'production'}),
    Document(page_content='Agent evaluation uses task completion rate, tool selection accuracy, reasoning quality, and cost per task. The evaluation flywheel: offline eval → deploy → monitor → collect failures → update test set → refine.', metadata={'source': 'evaluation.txt', 'topic': 'evaluation'}),
]

# Create vector store
vectorstore = Chroma.from_documents(knowledge_base, embedding=embedder)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
print(f'Knowledge base: {vectorstore._collection.count()} documents indexed')

Knowledge base: 10 documents indexed


## 7.2 From Static RAG to Agentic RAG

| Aspect | Static RAG | Agentic RAG |
|--------|-----------|-------------|
| **Query** | User query → retrieve → answer | Agent decides *when* and *what* to retrieve |
| **Retrieval** | Always retrieves | Retrieves only when needed |
| **Iteration** | Single pass | Can re-query with improved terms |
| **Evaluation** | None | Agent evaluates if results are sufficient |
| **Tools** | Only retrieval | Retrieval + other tools combined |

The key insight: in agentic RAG, **retrieval is just another tool** the agent can choose to use.

## 7.3 Retrieval as a Tool

In [ ]:
from langchain_core.tools import tool

@tool
def search_knowledge_base(query: str) -> str:
    """Search ONLY the local AI Agents knowledge base"""
    try:
        docs = retriever.invoke(query)
    except Exception as exc:
        return f"SEARCH_ERROR: {type(exc).__name__}: {exc}"

    if not docs:
        return (
            "NO_RESULTS: No relevant documents were found in the local knowledge base"
            f"for query: {query}"
        )
    
    results = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        topic = doc.metadata.get("topic", "unknown")
        content = doc.page_content.strip()

        results.append(
            f"[{i}] Source: `{source}` \n\n Topic: `{topic}`\n\n"
            f"{content}"
        )
    
    return "FOUND_RESULTS:\n\n" + "\n\n".join(results)

In [7]:
from IPython.display import display, Markdown

result = search_knowledge_base.invoke({"query": "Langgraph features"})
display(Markdown(result))

FOUND_RESULTS:

[1] Source: `frameworks.txt` 

 Topic: `langgraph`

LangGraph is a framework by LangChain for building stateful, graph-based agent workflows. It uses StateGraph with nodes and edges for agent orchestration. Key features: conditional edges, checkpointing, human-in-the-loop.

[2] Source: `patterns.txt` 

 Topic: `memory`

Agent memory includes short-term (conversation history), long-term (persisted facts), and episodic (past experience). LangGraph supports checkpointing with SQLite and MemorySaver.

[3] Source: `patterns.txt` 

 Topic: `tools`

Function calling allows LLMs to invoke external tools by generating structured JSON describing the function name and arguments. OpenAI, Anthropic, and Google all support this pattern.

## 7.4 Building a RAG-Powered Agent with LangGraph

<img src="images/rag-powered-agent.png" width="50%" style="border-radius:10px;margin:12px 0;"/>

In [8]:
from typing import Literal

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import MessagesState
from langchain_core.messages import ToolMessage, HumanMessage, SystemMessage

# Tools available to the agent
tools = [search_knowledge_base]
llm_with_tools = llm.bind_tools(tools)

MAX_RETRIEVALS = 2

def count_retrievals(state: MessagesState) -> int:
    """Count how many tool observations exist in the current graph state"""
    return sum(isinstance(message, ToolMessage) for message in state['messages'])

def agent(state: MessagesState) -> dict:
    retrievals = count_retrievals(state)

    if retrievals >= MAX_RETRIEVALS:
        system = SystemMessage(content=(
            "You are an AI agents expert"
            "You have already searched the local knowledge base."
            "Do not call tools anymore"
            "If the retrieved context is missing, unrelated or contains NO_RESULTS"
            "say clearly that the knowledge base does not contain enough information,"
            "Do not invent facts or sources."
        ))
        response = llm.invoke([system] + state['messages'])
    else:
        # Tool enale pass
        system = SystemMessage(content=(
            "You are an AI agents expert."
            "Use the search_knowledge_base tool for questions about AI-agent frameworks,"
            "patterns, protocols, production practices and evaluation."
            "Always cite retrieved sources using their Source labels."
            "If the tool returns NO_RESULTS or unrelated context, do not keep searching forever"
            "answer that the knowledge base is insufficient."
        ))
        response = llm_with_tools.invoke([system] + state["messages"])
    
    return {"messages":[response]}

In [9]:
def should_retrive(state: MessagesState) -> Literal["retrive", "__end__"]:
    last = state['messages'][-1]
    retrievals = count_retrievals(state)

    if getattr(last, "tool_calls", None) and retrievals < MAX_RETRIEVALS:
        return "retrieve"
    
    return END

In [11]:
# Build graph

rag_graph = StateGraph(MessagesState)

rag_graph.add_node("agent", agent)
rag_graph.add_node("retrieve", ToolNode(tools))

rag_graph.add_edge(START, "agent")
rag_graph.add_conditional_edges(
    "agent",
    should_retrive,
    {
        "retrieve": "retrieve",
        END: END
    },
)
rag_graph.add_edge("retrieve", "agent")

rag_graph_app = rag_graph.compile()


In [12]:
# Test the explicit LangGraph RAG agent
questions = [
    "What is LangGraph and what are its key features?",
    "How does MCP differ from A2A protocol?",
    "What patterns are used for agent evaluation in production?",
    "What is the main purpose of Datanous AI company?",
]

for q in questions:
    print(f"\n{'='*60}")
    display(Markdown(f"**Q: {q}**"))

    result = rag_graph_app.invoke(
        {"messages": [HumanMessage(content=q)]},
        config={"recursion_limit": 8},
    )

    display(Markdown(f"A: {result['messages'][-1].content}"))

**Q: What is LangGraph and what are its key features?**

A: LangGraph is a framework developed by LangChain for creating stateful, graph-based agent workflows. Here are its key features:

1. **StateGraph**: Utilizes nodes and edges for orchestrating agents, allowing for complex workflows.
2. **Conditional Edges**: Supports dynamic decision-making within workflows based on specific conditions.
3. **Checkpointing**: Enables saving the state of workflows, which can be useful for resuming processes or for debugging.
4. **Human-in-the-Loop**: Facilitates human intervention in the workflow, allowing for adjustments and oversight as needed.

These features make LangGraph a powerful tool for building sophisticated AI agent systems. [Source: frameworks.txt]

**Q: How does MCP differ from A2A protocol?**

A: MCP (Model Context Protocol) and A2A (Agent-to-Agent Protocol) are both protocols used in the context of AI agents, but they serve different purposes and have distinct architectures.

1. **MCP (Model Context Protocol)**:
   - Developed by Anthropic, MCP standardizes how large language models (LLMs) connect to external tools and data.
   - It employs a client-server architecture and includes three main components: Tools, Resources, and Prompts. This structure facilitates the interaction between the model and external resources.

2. **A2A (Agent-to-Agent Protocol)**:
   - Created by Google, A2A focuses on enabling communication between different agents.
   - It includes components such as Agent Cards (for capability discovery), Executors (for task processing), and Artifacts (which represent work products). This protocol is designed to facilitate collaboration and task execution among multiple agents.

In summary, MCP is primarily about connecting LLMs to external resources, while A2A is centered on communication and collaboration between agents.

**Q: What patterns are used for agent evaluation in production?**

A: In production, agent evaluation typically involves several key patterns:

1. **Task Completion Rate**: This measures how effectively the agent completes assigned tasks.
2. **Tool Selection Accuracy**: Evaluates how accurately the agent selects the appropriate tools for the tasks at hand.
3. **Reasoning Quality**: Assesses the quality of the agent's reasoning processes in arriving at conclusions or actions.
4. **Cost per Task**: Analyzes the cost associated with completing each task, which is crucial for optimizing resource usage.

Additionally, there is an evaluation flywheel process that includes:
- **Offline Evaluation**: Testing the agent in a controlled environment before deployment.
- **Deployment**: Launching the agent into a production environment.
- **Monitoring**: Continuously observing the agent's performance.
- **Collecting Failures**: Gathering data on any failures or issues encountered.
- **Updating Test Set**: Modifying the test set based on collected data to improve future evaluations.
- **Refining**: Iteratively improving the agent based on insights gained from monitoring and failures.

These patterns help ensure that agents are effective and efficient in real-world applications. [Source: evaluation.txt]

**Q: What is the main purpose of Datanous AI company?**

A: The knowledge base does not contain specific information about the main purpose of Datanous AI company. Therefore, I cannot provide an answer regarding their objectives or mission.

### 7.4.2 Same implementation using create_agent

```                                     
    User question                        
    ↓                                    
    LLM / agent                          
    ↓                                    
    Αποφασίζει αν χρειάζεται tool call   
    ↓                                    
    Αν ναι → καλεί search_knowledge_base 
    ↓                                    
    Παίρνει retrieved context            
    ↓                                    
    Επιστρέφει ξανά στο LLM              
    ↓                                    
    Τελική απάντηση                      
```


- Το **create_agent** είναι high-level factory.
- Χτίζει έτοιμη agent αρχιτεκτονική πάνω σε LangGraph.
- Το LLM αποφασίζει αν θα καλέσει το retrieval tool.
- Αν το καλέσει, το αποτέλεσμα επιστρέφει ξανά στον agent.
- Αν δεν το καλέσει, ο agent απαντά και τερματίζει.

In [13]:
# Agentic RAG με Factory Pattern
# To retieval γίνεται tool και το create_agent() χτίζει έτοιμο LangGraph-based agent runtime

from langchain.agents import create_agent

rag_factory_app = create_agent(
    model=llm,
    tools=[search_knowledge_base],
    system_prompt=(
        "You are an AI agents expert. "
        "Use the search_knowledge_base tool for questions about AI-agent frameworks, "
        "patterns, protocols, production practices, and evaluation. "
        "Always cite retrieved sources using their Source labels. "
        "If the tool returns NO_RESULTS or unrelated context, say that the local knowledge base "
        "does not contain enough information. Do not invent facts or sources."
    )
)

In [14]:
# Test the factory RAG agent
questions = [
    "What is LangGraph and what are its key features?",
    "How does MCP differ from A2A protocol?",
    "What patterns are used for agent evaluation in production?",
    "What is the main purpose of Datanous AI company?",
    "Which is the capital of France?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")

    result = rag_factory_app.invoke(
        {"messages": [HumanMessage(content=q)]},
        config={"recursion_limit": 8},
    )

    display(Markdown(f"A: {result['messages'][-1].content}"))



Q: What is LangGraph and what are its key features?


A: LangGraph is a framework developed by LangChain for creating stateful, graph-based agent workflows. Here are its key features:

1. **StateGraph**: Utilizes nodes and edges for orchestrating agents, allowing for complex workflows.
2. **Conditional Edges**: Supports dynamic decision-making within the workflow based on specific conditions.
3. **Checkpointing**: Enables saving the state of the workflow, which can be useful for resuming processes or maintaining continuity.
4. **Human-in-the-Loop**: Facilitates human intervention in the workflow, allowing for oversight and adjustments as needed.

These features make LangGraph a powerful tool for building sophisticated AI agent systems. [Source: frameworks.txt]


Q: How does MCP differ from A2A protocol?


A: The MCP (Model Context Protocol) and A2A (Agent-to-Agent Protocol) are both protocols used in AI agent frameworks, but they serve different purposes and have distinct architectures.

1. **MCP (Model Context Protocol)**:
   - Developed by Anthropic, MCP standardizes how large language models (LLMs) connect to external tools and data.
   - It employs a client-server architecture and consists of three main components: Tools, Resources, and Prompts. This structure allows for efficient interaction between the model and external resources, facilitating the integration of various functionalities.

2. **A2A (Agent-to-Agent Protocol)**:
   - Created by Google, A2A focuses on enabling communication between different agents.
   - Its components include Agent Cards (for capability discovery), Executors (for task processing), and Artifacts (which represent work products). This protocol is designed to facilitate collaboration and task delegation among multiple agents.

In summary, while MCP is centered around the interaction of LLMs with external tools, A2A is focused on the communication and collaboration between agents themselves.


Q: What patterns are used for agent evaluation in production?


A: In production, agent evaluation typically involves several key patterns, including:

1. **Task Completion Rate**: This measures how effectively the agent completes assigned tasks.
2. **Tool Selection Accuracy**: This evaluates how accurately the agent selects the appropriate tools for the tasks at hand.
3. **Reasoning Quality**: This assesses the quality of the agent's reasoning processes during task execution.
4. **Cost per Task**: This looks at the financial efficiency of the agent in completing tasks.

Additionally, there is an evaluation flywheel process that includes:
- Offline evaluation
- Deployment
- Monitoring
- Collecting failures
- Updating the test set
- Refining the evaluation criteria

These patterns help ensure that agents are not only effective but also continuously improving based on real-world performance data (Source: `evaluation.txt`).


Q: What is the main purpose of Datanous AI company?


A: The local knowledge base does not contain enough information about the main purpose of Datanous AI company.


Q: Which is the capital of France?


A: The local knowledge base does not contain enough information to answer that question.

### 7.4.3. Dimitris is here! :D

In [15]:
# Agentic RAG με factory pattern:
# Το retrieval γίνεται @tool και το create_agent χτίζει έτοιμο LangGraph-based agent runtime.
from langchain.agents import create_agent

rag_factory_app = create_agent(
    model=llm,
    tools=[search_knowledge_base],
    system_prompt=(
        "You are an AI agents expert. "
        "Use the search_knowledge_base tool for questions about AI-agent frameworks, "
        "patterns, protocols, production practices, and evaluation. "
        "Always cite retrieved sources using their Source labels. "
        "If the tool returns NO_RESULTS or unrelated context, "
        "answer based on your own knowledge BUT clearly state: "
        "'Note: The local knowledge base had no matching documents; "
        "this answer is based on general training knowledge.'"
    ),
)

In [16]:
# Test the factory RAG agent
questions = [
    "What is LangGraph and what are its key features?",
    "How does MCP differ from A2A protocol?",
    "What patterns are used for agent evaluation in production?",
    "What is the main purpose of Datanous AI company?",
    "Which is the capital of France?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")

    result = rag_factory_app.invoke(
        {"messages": [HumanMessage(content=q)]},
        config={"recursion_limit": 8},
    )

    display(Markdown(f"A: {result['messages'][-1].content}"))



Q: What is LangGraph and what are its key features?


A: LangGraph is a framework developed by LangChain designed for building stateful, graph-based agent workflows. Here are its key features:

1. **StateGraph**: Utilizes a graph structure with nodes and edges to orchestrate agent workflows effectively.
2. **Conditional Edges**: Allows for dynamic decision-making within the workflow based on specific conditions.
3. **Checkpointing**: Supports saving the state of the workflow, enabling recovery and continuity in processes.
4. **Human-in-the-Loop**: Facilitates human intervention in the workflow, allowing for oversight and adjustments as needed.

These features make LangGraph a powerful tool for creating complex agent interactions and workflows. [Source: frameworks.txt]


Q: How does MCP differ from A2A protocol?


A: The MCP (Model Context Protocol) and A2A (Agent-to-Agent Protocol) are both frameworks designed for facilitating communication and interaction among AI agents, but they serve different purposes and have distinct architectures.

### MCP (Model Context Protocol)
- **Purpose**: MCP is designed to standardize how large language models (LLMs) connect to external tools and data.
- **Architecture**: It employs a client-server architecture and is built around three main primitives:
  - **Tools**: External functionalities that the LLM can utilize.
  - **Resources**: Data or information that the LLM can access.
  - **Prompts**: Instructions or queries that guide the LLM's interactions with tools and resources.
- **Source**: [protocols.txt]

### A2A (Agent-to-Agent Protocol)
- **Purpose**: A2A focuses on enabling communication between different agents, allowing them to collaborate and share tasks.
- **Components**:
  - **Agent Cards**: Used for capability discovery among agents.
  - **Executors**: Responsible for processing tasks assigned to agents.
  - **Artifacts**: The work products generated by agents during their interactions.
- **Source**: [protocols.txt]

### Key Differences
1. **Focus**: MCP is centered on LLMs interacting with external tools and data, while A2A is about communication and collaboration between multiple agents.
2. **Architecture**: MCP uses a client-server model, whereas A2A involves a more decentralized approach where agents can discover capabilities and execute tasks collaboratively.

These differences highlight the distinct roles each protocol plays in the ecosystem of AI agents.


Q: What patterns are used for agent evaluation in production?


A: In production, agent evaluation typically employs several key patterns, including:

1. **Task Completion Rate**: This measures how effectively the agent completes assigned tasks.
2. **Tool Selection Accuracy**: This evaluates how accurately the agent selects the appropriate tools for the tasks at hand.
3. **Reasoning Quality**: This assesses the quality of the agent's reasoning processes in arriving at conclusions or actions.
4. **Cost per Task**: This looks at the efficiency of the agent in terms of resource utilization for each task it performs.

Additionally, a structured evaluation process often follows an iterative cycle known as the evaluation flywheel, which includes:
- **Offline Evaluation**: Testing the agent in a controlled environment before deployment.
- **Deployment**: Launching the agent into a live environment.
- **Monitoring**: Continuously observing the agent's performance.
- **Collecting Failures**: Gathering data on any failures or issues encountered.
- **Updating the Test Set**: Modifying the evaluation criteria based on observed performance.
- **Refining**: Improving the agent based on insights gained from the evaluation process.

These patterns help ensure that agents are not only effective but also continuously improving over time (Source: `evaluation.txt`).


Q: What is the main purpose of Datanous AI company?


A: Note: The local knowledge base had no matching documents; this answer is based on general training knowledge.

Datanous AI is primarily focused on providing AI-driven solutions that enhance data management and analytics. Their main purpose is to leverage artificial intelligence to help organizations make better data-driven decisions, optimize their operations, and improve overall efficiency. They typically offer tools and platforms that facilitate data integration, analysis, and visualization, enabling businesses to harness the power of their data effectively.


Q: Which is the capital of France?


A: Note: The local knowledge base had no matching documents; this answer is based on general training knowledge. The capital of France is Paris.

### 7.4.4. Explicitly detext tool calls

In [17]:
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage

def inspect_agent_run(agent, question: str):
    result = agent.invoke({
        "messages": [HumanMessage(content=question)]
    })

    print("\n" + "=" * 80)
    print(f"QUESTION: {question}")
    print("=" * 80)

    tool_used = False

    print("\nFULL MESSAGE TRACE")
    print("-" * 80)

    for i, msg in enumerate(result["messages"]):
        print(f"\n[{i}] {msg.__class__.__name__}")
        msg.pretty_print()

    print("\nTOOL USAGE SUMMARY")
    print("-" * 80)

    for msg in result["messages"]:

        if isinstance(msg, AIMessage):
            tool_calls = getattr(msg, "tool_calls", None)

            if tool_calls:
                tool_used = True

                for call in tool_calls:
                    print("LLM requested tool:")
                    print(f"  Tool name : {call.get('name')}")
                    print(f"  Arguments : {call.get('args')}")
                    print(f"  Call ID   : {call.get('id')}")
                    print()

        elif isinstance(msg, ToolMessage):
            tool_used = True

            print("Tool returned result:")
            print(f"  Tool name    : {getattr(msg, 'name', 'unknown')}")
            print(f"  Tool call ID : {getattr(msg, 'tool_call_id', 'unknown')}")
            print("  Content:")
            print(msg.content[:1000])
            print()

    print(f"TOOL USED? {tool_used}")

    final_answer = result["messages"][-1].content

    print("\nFINAL ANSWER")
    print("-" * 80)
    print(final_answer)

    return result

inspect_agent_run(
    rag_factory_app,
    "What is LangGraph and what are its key features?"
    # "Which is the capital of France?"
)


QUESTION: What is LangGraph and what are its key features?

FULL MESSAGE TRACE
--------------------------------------------------------------------------------

[0] HumanMessage
================================ Human Message =================================

What is LangGraph and what are its key features?

[1] AIMessage
================================== Ai Message ==================================
Tool Calls:
  search_knowledge_base (call_PLvtFY5JejbNSXor35qVILQ4)
 Call ID: call_PLvtFY5JejbNSXor35qVILQ4
  Args:
    query: LangGraph

[2] ToolMessage
================================= Tool Message =================================
Name: search_knowledge_base

FOUND_RESULTS:

[1] Source: `frameworks.txt` 

 Topic: `langgraph`

LangGraph is a framework by LangChain for building stateful, graph-based agent workflows. It uses StateGraph with nodes and edges for agent orchestration. Key features: conditional edges, checkpointing, human-in-the-loop.

[2] Source: `patterns.txt` 

 Topic: `m

{'messages': [HumanMessage(content='What is LangGraph and what are its key features?', additional_kwargs={}, response_metadata={}, id='dd4fee62-f448-4c4b-af05-945f4b458523'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 140, 'total_tokens': 157, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d273f354a1', 'id': 'chatcmpl-Di1gUWZTg8nwujk8zlXWcJggeVPTy', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e4ba0-a3b9-7931-99da-22a3b18aded0-0', tool_calls=[{'name': 'search_knowledge_base', 'args': {'query': 'LangGraph'}, 'id': 'call_PLvtFY5JejbNSXor35qVILQ4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_m

## 7.5 Iterative Retrieval — The Maker-Checker Pattern

In agentic RAG, the agent can **evaluate** whether retrieved documents are sufficient and re-query if needed:

<img src="images/iterative-retrieval.png" width="50%" style="border-radius:10px;margin:12px 0;"/>

This self-correcting behavior is what makes agentic RAG superior to static RAG.

In [18]:
from pydantic import BaseModel, Field

class RetrievalEvaluation(BaseModel):
    """Evaluate whether retrieved documents are sufficient."""
    is_sufficient: bool = Field(description="Whether the documents fully answer the question")
    reasoning: str = Field(description="Why the documents are or are not sufficient")
    improved_query: str = Field(default="", description="Better search query if not sufficient")

evaluator = llm.with_structured_output(RetrievalEvaluation)


def format_docs(docs: list[Document]) -> str:
    """Format documents with metadata so the final answer can cite sources."""
    formatted = []

    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        topic = doc.metadata.get("topic", "unknown")
        formatted.append(
            f"[{i}] Source: {source} | Topic: {topic}\n"
            f"{doc.page_content.strip()}"
        )

    return "\n\n".join(formatted)


def dedupe_docs(docs: list[Document]) -> list[Document]:
    """Remove duplicate documents while preserving order."""
    seen = set()
    unique_docs = []

    for doc in docs:
        key = (
            doc.metadata.get("source", "unknown"),
            doc.metadata.get("topic", "unknown"),
            doc.page_content,
        )

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    return unique_docs


def iterative_rag(question: str, max_attempts: int = 3) -> str:
    """RAG with self-evaluation and query refinement."""
    query = question
    all_docs = []
    final_evaluation = None

    for attempt in range(1, max_attempts + 1):
        print(f"\n  Attempt {attempt}: Searching for {query!r}")

        docs = retriever.invoke(query)
        all_docs.extend(docs)
        unique_docs = dedupe_docs(all_docs)

        if not docs:
            print("  No documents retrieved.")
            break

        doc_text = format_docs(docs)

        evaluation = evaluator.invoke([
            SystemMessage(content=(
                "Evaluate whether the retrieved documents are sufficient to answer the question. "
                "If a required entity or concept is missing from the documents, mark is_sufficient=false. "
                "If you suggest an improved query, make it meaningfully different from the current one."
            )),
            HumanMessage(content=(
                f"Question: {question}\n"
                f"Current query: {query}\n\n"
                f"Retrieved documents:\n{doc_text}"
            )),
        ])

        final_evaluation = evaluation

        display(Markdown(f"  Sufficient: {evaluation.is_sufficient}"))
        display(Markdown(f"  Reasoning: {evaluation.reasoning}"))

        if evaluation.is_sufficient:
            break

        if not evaluation.improved_query or evaluation.improved_query.strip() == query.strip():
            display(Markdown("  No useful refined query produced. Stopping."))
            break

        query = evaluation.improved_query.strip()
        display(Markdown(f"  Refined query: {query}"))

    unique_docs = dedupe_docs(all_docs)

    if not unique_docs:
        return (
            "I could not find relevant information in the local knowledge base, "
            "so I cannot answer this reliably from the provided sources."
        )

    all_context = format_docs(unique_docs)

    if final_evaluation and not final_evaluation.is_sufficient:
        answer_policy = (
            "The retrieved context may be incomplete. "
            "Answer only the parts supported by the context and explicitly mention missing information."
        )
    else:
        answer_policy = "Answer based only on the provided context."

    response = llm.invoke([
        SystemMessage(content=(
            f"{answer_policy} "
            "Cite sources using the exact Source labels from the context. "
            "Do not invent missing facts."
        )),
        HumanMessage(content=f"Context:\n{all_context}\n\nQuestion: {question}")
    ])

    return response.content


answer = iterative_rag("Compare the multi-agent capabilities of LangGraph, CrewAI, and AutoGen")
display(Markdown(f"\nFinal Answer:\n{answer}"))



  Attempt 1: Searching for 'Compare the multi-agent capabilities of LangGraph, CrewAI, and AutoGen'


  Sufficient: True

  Reasoning: The retrieved documents provide a clear comparison of the multi-agent capabilities of LangGraph, CrewAI, and AutoGen. Each document outlines the specific features and functionalities of the frameworks, allowing for a comprehensive understanding of their multi-agent capabilities.


Final Answer:
LangGraph, CrewAI, and AutoGen each offer unique multi-agent capabilities tailored to different use cases.

1. **LangGraph**: This framework focuses on building stateful, graph-based agent workflows. It utilizes a StateGraph structure with nodes and edges for orchestrating agents. Key features include conditional edges, checkpointing, and the ability to incorporate human-in-the-loop interactions, which allows for dynamic decision-making and adaptability in agent workflows [1].

2. **CrewAI**: In contrast, CrewAI is designed for role-based multi-agent systems where each agent has a specific role, goal, and backstory. This framework emphasizes collaboration among agents through structured task delegation and shared context, making it suitable for scenarios where agents need to work together towards common objectives while maintaining distinct identities and responsibilities [2].

3. **AutoGen**: Developed by Microsoft, AutoGen facilitates multi-agent conversation patterns, supporting features like group chat, agent handoffs, and distributed execution via gRPC. This allows agents to operate on separate machines, enhancing scalability and flexibility in communication and task execution among agents [3].

In summary, LangGraph excels in orchestrating complex workflows with state management, CrewAI focuses on collaborative role-based interactions, and AutoGen emphasizes conversational dynamics and distributed execution. Each framework serves different needs in the realm of multi-agent systems.

In [21]:
answer2 = iterative_rag(
    # "How does LangGraph differ from CrewAI in multi-agent orchestration?"
    "Compare the multi-agent capabilities of LangGraph, CrewAI, and Datanous"
)
display(Markdown(f"\nFinal Answer:\n{answer2}"))


  Attempt 1: Searching for 'Compare the multi-agent capabilities of LangGraph, CrewAI, and Datanous'


  Sufficient: False

  Reasoning: The retrieved documents provide information on LangGraph and CrewAI, but they do not include any details about Datanous, which is essential for a complete comparison. Additionally, while the documents describe the capabilities of LangGraph and CrewAI, they lack a direct comparison or analysis of how these frameworks stack up against each other in terms of multi-agent capabilities.

  Refined query: Compare the multi-agent capabilities of LangGraph, CrewAI, and Datanous, including specific features and use cases.


  Attempt 2: Searching for 'Compare the multi-agent capabilities of LangGraph, CrewAI, and Datanous, including specific features and use cases.'


...


  Attempt 3: Searching for 'Compare the multi-agent capabilities, features, and use cases of LangGraph, CrewAI, and Datanous, including any unique aspects of each framework.'


  Sufficient: False

  Reasoning: The retrieved documents provide some information about LangGraph and CrewAI, but they do not include any details about Datanous, which is essential for a complete comparison. Additionally, while the documents mention some features and capabilities, they lack a comprehensive overview of use cases and unique aspects for each framework, particularly for Datanous.

  Refined query: Compare the multi-agent capabilities, features, use cases, and unique aspects of LangGraph, CrewAI, and Datanous, including detailed descriptions of each framework.


Final Answer:
The context provides information on the multi-agent capabilities of LangGraph and CrewAI, but does not include any details about Datanous. 

1. **LangGraph**: This framework focuses on building stateful, graph-based agent workflows. It utilizes a StateGraph with nodes and edges for agent orchestration, allowing for features like conditional edges, checkpointing, and human-in-the-loop interactions. However, the specific multi-agent capabilities are not detailed in the context.

2. **CrewAI**: This framework is designed for role-based multi-agent systems, where each agent has a defined role, goal, and backstory. Agents collaborate through structured task delegation and shared context, emphasizing teamwork and collaboration among agents.

3. **Datanous**: There is no information provided about Datanous in the context, so its multi-agent capabilities cannot be compared.

In summary, while LangGraph and CrewAI have distinct approaches to multi-agent systems, the lack of information on Datanous prevents a complete comparison.

## 💡 Exercise 7: Build a RAG Agent with Multiple Knowledge Bases

**Task**: Build a RAG agent that:
1. Has two separate knowledge bases (e.g., "frameworks" and "best practices")
2. Uses a router to decide which knowledge base to query
3. Can combine information from both in a single answer

Hint: Create two `@tool` functions, each querying a different Chroma collection.

In [ ]:
# Exercise 7: YOUR CODE HERE


## 📝 Summary

| Concept | Key Takeaway |
|---------|-------------|
| **Agentic RAG** | Agent decides when/what to retrieve, not a fixed pipeline |
| **Retrieval as tool** | Wrap vector search in `@tool` for agent access |
| **Iterative retrieval** | Evaluate → re-query → collect until sufficient |
| **Maker-checker** | Generate answer, check quality, improve if needed |

### What's Next

In **Notebook 08: ReAct and Tool-Using Agents**, we implement the full ReAct pattern — the most popular agent architecture for combining reasoning with tool use.